In [1]:
import pandas as pd
import duckdb

df_device_status_daily = pd.DataFrame({
    "device_id": [
        "A", "A", "A", "A", "A", "A", "A",
        "B", "B", "B", "B", "B",
        "C", "C", "C", "C"
    ],
    "stat_date": [
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04",
        "2026-07-05",
        "2026-07-07",
        "2026-07-08",
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04",
        "2026-07-06",
        "2026-07-01",
        "2026-07-03",
        "2026-07-04",
        "2026-07-05"
    ],
    "status": [
        "OK",
        "ERROR",
        "ERROR",
        "OK",
        "ERROR",
        "ERROR",
        "ERROR",
        "ERROR",
        "ERROR",
        "ERROR",
        "OK",
        "ERROR",
        "OK",
        "ERROR",
        "ERROR",
        "OK"
    ]
})

df_device_status_daily["stat_date"] = pd.to_datetime(
    df_device_status_daily["stat_date"]
)

df_device_status_daily

,device_id,stat_date,status
0,A,2026-07-01,OK
1,A,2026-07-02,ERROR
2,A,2026-07-03,ERROR
3,A,2026-07-04,OK
4,A,2026-07-05,ERROR
5,A,2026-07-07,ERROR
6,A,2026-07-08,ERROR
7,B,2026-07-01,ERROR
8,B,2026-07-02,ERROR
9,B,2026-07-03,ERROR


## 题目要求

找出每台设备中，**连续至少 2 个自然日处于 `ERROR` 状态的异常区间**。

### 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `error_start_date` | 连续异常开始日期 |
| `error_end_date` | 连续异常结束日期 |
| `error_days` | 连续异常自然日数 |

### 排序要求

最终结果按照以下顺序排列：

1. `device_id` 升序
2. `error_start_date` 升序

### 连续异常的判断规则

连续异常必须满足：

```text
当前异常日期 = 上一条异常日期 + 1 天

In [15]:
query = """
WITH error_with_previous_date AS (
    SELECT
        *,
        LAG(stat_date) OVER (
            PARTITION BY device_id
            ORDER BY stat_date
        ) AS previous_date
    FROM df_device_status_daily
    WHERE status = 'ERROR'
),

error_start AS (
    SELECT
        device_id,
        stat_date,
        status,
        previous_date,
        CASE
            WHEN previous_date IS NULL
                 OR stat_date > previous_date + INTERVAL 1 DAY
            THEN 1
            ELSE 0
        END AS start_sign
    FROM error_with_previous_date
),

error_phase AS (
    SELECT
        *,
        SUM(start_sign) OVER (
            PARTITION BY device_id
            ORDER BY stat_date
        )::INTEGER AS phase_sign
    FROM error_start
)

SELECT
    device_id,
    MIN(stat_date) AS error_start_date,
    MAX(stat_date) AS error_end_date,
    COUNT(*) AS error_days
FROM error_phase
GROUP BY
    device_id,
    phase_sign
HAVING COUNT(*) >= 2
ORDER BY
    device_id,
    error_start_date;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,error_start_date,error_end_date,error_days
0,A,2026-07-02,2026-07-03,2
1,A,2026-07-07,2026-07-08,2
2,B,2026-07-01,2026-07-03,3
3,C,2026-07-03,2026-07-04,2
